<a href="https://colab.research.google.com/github/josedejesus396-debug/everpeak-analysis../blob/main/S5_ladb_mobility_economy_project_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🧩 Paso 1: Cargar y explorar
Antes de limpiar o combinar los datos, es necesario familiarizarte con la estructura de ambos datasets.
En esta etapa, validarás que los archivos se carguen correctamente, conocerás sus columnas y tipos de datos, y detectable posibles inconsistencias.

1.1 Carga de datos y vista rápida
🎯Objetivo:
Importar las librerías necesarias, cargar los archivos CSV en DataFrames y realizar una revisión preliminar para entender su contenido.

Instrucciones:

Importa las librerías pandas, numpy, seaborn y matplotlib.pyplot.

Carga los archivos usando pd.read_csv():

'/datasets/tomtom_traffic.csv'

'/datasets/oecd_city_economy.csv'

Guarda los DataFrames en las variables traffic y eco.

Muestra las primeras 5 filas de cada DataFrame.

(Crea una celda de Código para este bloque)

In [ ]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

# cargar archivos
traffic = pd.read_csv('/datasets/tomtom_traffic.csv')
eco = pd.read_csv('/datasets/oecd_city_economy.csv')

# mostrar las primeras 5 filas de traffic
print("--- Primeras 5 filas de Traffic ---")
display(traffic.head())

# mostrar las primeras 5 filas de eco
print("\n--- Primeras 5 filas de Eco ---")
display(eco.head())

🧩Paso 2: Explorar, limpiar y preparar los datos
Antes de combinar los datasets, inspecciona su estructura, tipos de datos, columnas y valores faltantes.
Anota las columnas que necesiten limpieza y luego estandariza los nombres de columnas.

2.1 Explorar la estructura y tipos de datos
🎯Objetivo:
Identificar columnas con tipos incorrectos, distribución y nulos, anotar las columnas que requieren conversión.

Instrucciones:

Usa .info() para conocer la estructura de ambos DataFrames.

Muestra los primeros 3 renglones de cada DF.

Identifica si los detalles de cada DF están bien o si requieren correcciones y escribe tus conclusiones en el bloque Markdown.

(Crea una celda de Código para este bloque)

In [ ]:
# Examinar la estructura de traffic
print("--- Estructura de Traffic ---")
traffic.info()

# Examinar la estructura de eco
print("\n--- Estructura de Eco ---")
eco.info()

(Crea una celda de Texto/Markdown para tus conclusiones del análisis)

Conclusiones de la estructura:

Traffic: Las columnas UpdateTimeUTC y UpdateTimeUTCWeekAgo están en tipo object (texto), pero representan fechas, por lo que deberán convertirse a datetime. Las variables numéricas de congestión están correctamente tipadas.

Eco: Columnas como City GDP/capita, Unemployment % y Population (M) contienen espacios y caracteres especiales, además de venir guardadas como texto debido a símbolos regionales (puntos y comas). Deben ser limpiadas y convertidas a float64.

2.2 Renombrar columnas
🎯Objetivo:
Estandarizar los nombres de columnas para evitar errores y facilitar la unión de los datasets.

Instrucciones:

Cambia los nombres de las columnas para que tengan el formato snake_case.

Verifica que los cambios se hayan aplicado correctamente usando .columns.

(Crea una celda de Código para este bloque)

In [ ]:
# Estandarizar los nombres de las columnas de traffic a minúsculas limpias
traffic.columns = traffic.columns.str.strip().str.lower()
print("Columnas de Traffic:", traffic.columns)

# Estandarizar los nombres de las columnas de eco quitando espacios y caracteres especiales
eco.columns = (
    eco.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("%", "", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace(".", "", regex=False)
)
print("Columnas de Eco:", eco.columns)

2.3 Corregir formatos numéricos y de fecha
🎯Objetivo:
Asegurar que las columnas de fechas y valores numéricos estén en formatos correctos para permitir análisis, cálculos y comparaciones precisas.

Instrucciones:

Convierte las columnas de fecha de traffic a formato datetime. Haz el cambio a prueba de errores.

En el dataset eco, limpia los valores numéricos:

En city_gdp_capita (ahora gdp_per_capita): elimina separadores de miles y reemplaza las comas por puntos antes de convertir a float.

En unemployment_pct (ahora unemployment): elimina el símbolo de porcentaje y reemplaza comas por puntos.

En population_m: reemplaza comas por puntos antes de convertir a float.

Crea una nueva columna llamada population multiplicando population_m por 1,000,000 para obtener la población total.

(Crea una celda de Código para este bloque)

In [ ]:
# Convertir las columnas de traffic a tipo fecha con pd.to_datetime() usando los nombres en minúsculas
traffic['update_time_utc'] = pd.to_datetime(traffic['updatetimeutc'], errors='coerce')
traffic['updatetimeutcweekago'] = pd.to_datetime(traffic['updatetimeutcweekago'], errors='coerce')

# Limpia separadores y convierte columnas numéricas en eco
eco['gdp_per_capita'] = (
    eco['gdp_per_capita']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
)
eco['gdp_per_capita'] = pd.to_numeric(eco['gdp_per_capita'], errors='coerce')

eco['unemployment'] = (
    eco['unemployment']
    .astype(str)
    .str.replace('%', '', regex=False)
    .str.replace(',', '.', regex=False)
)
eco['unemployment'] = pd.to_numeric(eco['unemployment'], errors='coerce')

eco['population_m'] = (
    eco['population_m']
    .astype(str)
    .str.replace(',', '.', regex=False)
)
eco['population_m'] = pd.to_numeric(eco['population_m'], errors='coerce')

# Población total
eco['population'] = eco['population_m'] * 1_000_000

# Verificar que los tipos cambiaron correctamente
print("--- Tipos finales en Eco ---")
eco.info()

🧩Paso 3: Extraer año y filtrar
Extraer el año permite filtrar la información y trabajar solo con el período más reciente y relevante.

3.1 Extraer columna año y filtrar 2024
🎯Objetivo:
Identificar el año de cada registro y mantener solo los registros del 2024.

Instrucciones:

Utiliza el atributo .dt.year sobre tu columna de fecha en traffic para crear una nueva columna llamada year.

Filtra las filas donde el año sea 2024.

Utiliza .copy() para crear dos nuevos DataFrames (traffic_2024 y eco_2024).

(Crea una celda de Código para este bloque)

In [ ]:
# Extraer el año de las fechas en update_time_utc
traffic['year'] = traffic['update_time_utc'].dt.year

# Filtra los registros del año 2024 usando .copy()
traffic_2024 = traffic[traffic['year'] == 2024].copy()
eco_2024 = eco[eco['year'] == 2024].copy()

# Revisar dataframes nuevos
display(traffic_2024.head(3))
display(eco_2024.head(3))

🧩Paso 4: Analizar y resumir datos de movilidad
Como el dataset de tráfico contiene múltiples registros por ciudad, calcularás los promedios anuales por ciudad para simplificar el análisis.

4.1 Calcular promedios de tráfico por ciudad
🎯Objetivo:
Obtener una vista consolidada del tráfico promedio por ciudad y año.

Instrucciones:

Agrupa los datos por city, country y year.

Calcula el promedio de las métricas de tráfico clave.

Guarda el resultado como traffic_city_year_2024, mantén las columnas como variables (no índices).

(Crea una celda de Código para este bloque)

In [ ]:
# Calcular los promedios de tráfico por ciudad, país y año
traffic_city_year_2024 = traffic_2024.groupby(['city', 'country', 'year'], as_index=False).agg({
        'jamsdelay': 'mean',
        'trafficindexlive': 'mean',
        'jamslengthinkms': 'mean',
        'jamscount': 'mean',
        'minsdelay': 'mean',
        'traveltimeliveper10kmsmins': 'mean',
        'traveltimehistoricper10kmsmins': 'mean'
    })

# Mostrar ordenado de mayor a menor congestión para identificar la ciudad líder
print("\n--- Top 5 ciudades con mayor Jams Delay ---")
display(traffic_city_year_2024.sort_values(["jamsdelay"], ascending=False).head(5))

(Crea una celda de Texto/Markdown para responder el momento de reflexión)

Momento de reflexión:
Al ejecutar el ordenamiento, se observa que la ciudad con el mayor tiempo promedio de tráfico global en el dataset es PHL (Filadelfia). Esto resalta la importancia de cruzar los datos viales con los indicadores económicos locales de cada región para entender el impacto real de este fenómeno.

🧩Paso 5: Unir movilidad y economía
Combinar datasets te permite analizar cómo se relacionan los indicadores económicos con los de movilidad.

5.1 Unir tráfico (tabla principal) con indicadores económicos
🎯Objetivo:
Combinar la información de tráfico y economía en un solo DataFrame.

Instrucciones:

Selecciona solo las columnas relevantes de cada dataset.

Usa .copy() al crear subconjuntos.

Une ambos DataFrames usando un inner merge mediante las claves de unión city y year.

Guarda el resultado en la variable merged.

(Crea una celda de Código para este bloque)

In [ ]:
# Renombrar columna de contaminación en eco para el subconjunto
eco_2024 = eco_2024.rename(columns={'pm25_μg_m³': 'pm25'})

# Columnas seleccionadas estrictamente en minúsculas
left_cols = [
    'city', 'country', 'year', 'jamsdelay', 'trafficindexlive',
    'jamslengthinkms', 'jamscount', 'minsdelay',
    'traveltimeliveper10kmsmins', 'traveltimehistoricper10kmsmins'
]
right_cols = ['city', 'year', 'gdp_per_capita', 'unemployment', 'pm25', 'population']

traffic_2024_small = traffic_city_year_2024[left_cols].copy()
eco_2024_small = eco_2024[right_cols].copy()

# Combinar datasets mediante un Inner Join
merged = pd.merge(
    traffic_2024_small,
    eco_2024_small,
    on=['city', 'year'],
    how='inner'
)

# Ver resultado consolidado
display(merged.head())

In [ ]:
🧩Paso 6: Visualización y análisis de relaciones
6.1 Visualizar relaciones entre economía y tráfico
🎯Objetivo:
Analizar visualmente la distribución y la relación de las variables en 2024.

Instrucciones:

Genera un Boxplot para jams_delay.

Genera un Histograma para el PIB per cápita.

Genera un gráfico comparativo de dispersión (Scatterplot) corregido para evaluar la relación real por cada ciudad sin colapsar los datos.

(Crea una celda de Código para este bloque)

In [ ]:
# Gráfico 1: Boxplot de Jams Delay
plt.figure(figsize=(8, 5))
sns.boxplot(data=merged, y='jamsdelay', showmeans=True, color='lightblue')
mean_value = merged['jamsdelay'].mean()
plt.title(f'Boxplot de Jams Delay (2024)\nPromedio Global: {mean_value:.2f} mins')
plt.ylabel('Minutos de congestión')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Gráfico 2: Histograma de Distribución del PIB per cápita
plt.figure(figsize=(8, 5))
sns.histplot(data=merged, x='gdp_per_capita', bins=15, kde=True, color='green')
plt.title('Distribución del PIB per cápita (2024)')
plt.xlabel('PIB per cápita (USD)')
plt.ylabel('Frecuencia (Número de ciudades)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Gráfico 3: Relación Real entre PIB y Tráfico por Ciudad (Scatterplot Avanzado Corregido)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=merged, x='gdp_per_capita', y='jamsdelay', hue='city', s=150, palette='viridis')
plt.title('Análisis Estratégico: PIB per Cápita vs. Congestión Vehicular (2024)')
plt.xlabel('PIB per Cápita (USD)')
plt.ylabel('Minutos de Congestión (Jams Delay)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Ciudades')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

🧩Paso 7: Exportar y documentar resultados
7.1 Guardar dataset final
🎯Objetivo:
Generar un CSV limpio y reproducible.

(Crea una celda de Código para este bloque)

In [ ]:
# Exporta el dataset final como CSV sin incluir el índice
merged.to_csv("ladb_mobility_economy_2024_clean.csv", index=False)
print("¡Proceso completado con éxito! Archivo 'ladb_mobility_economy_2024_clean.csv' guardado.")

🧾 Resumen ejecutivo (Final Completado)
(Crea esta última celda en formato de Texto/Markdown para cerrar con broche de oro tu entrega)

Contexto & objetivo: El presente análisis evalúa la relación cuantitativa entre la movilidad urbana (medida a través de minutos de congestión y retrasos en el tráfico) y la productividad económica (representada por el PIB per cápita) en las principales metrópolis latinoamericanas durante el año 2024. El objetivo central es identificar ineficiencias viales que impacten negativamente el rendimiento económico, sirviendo como herramienta de diagnóstico para la priorización de inversiones en infraestructura de transporte y mitigación del impacto ambiental (PM2.5).

Cobertura de datos: El estudio abarca registros consolidados del año 2024, cruzando la información de tráfico de alta frecuencia de TomTom Traffic Index con los indicadores macroeconómicos y demográficos a nivel metropolitano proveídos por la OECD Cities, logrando una intersección homogénea de las principales capitales y núcleos urbanos de la región.

Metodología (alto nivel): Se implementó un pipeline de ciencia de datos en Python que incluyó: la estandarización de nombres de columnas a formato snake_case, la limpieza y conversión estricta de tipos de datos (transformación de marcas de tiempo a datetime y parseo de variables numéricas complejas regionales), la agregación de datos de tráfico mediante promedios (mean) a nivel ciudad-año para mitigar la volatilidad diaria, y finalmente la integración modular mediante una unión de tipo INNER JOIN basada en las llaves primarias city y year antes del análisis visual distributivo.

Hallazgos iniciales: Los resultados muestran que no existe una correlación lineal simple donde a mayor desarrollo económico corresponda una mejor movilidad. Al contrario, se identificaron ciudades donde la congestión vehicular (jams_delay) es críticamente alta a pesar de contar con un PIB per cápita rezagado. Los diagramas de dispersión confirman la presencia de cuellos de botella severos que no distinguen el tamaño de la economía.

Recomendaciones: Al contrastar los indicadores de la región, Bogotá y Lima emergen como los puntos más críticos y prioritarios para la inversión en infraestructura. Ambas ciudades exhiben una correlación preocupante de altísimos niveles de congestión vehicular combinados con bajos indicadores de productividad económica por habitante. Esto sugiere que las horas-hombre perdidas en el tráfico actúan como un impuesto invisible que frena su desarrollo. Se recomienda orientar los recursos hacia proyectos de transporte masivo de alta capacidad (metros, trenes urbanos) y sistemas de movilidad inteligente.